# Langbridge SDK Federated Query Notebook

This walkthrough shows the local Langbridge runtime federating three live sources at query time:

- a Postgres sales database
- a Postgres CRM database
- a local CSV with campaign attribution tags

The shared key is `contact_external_id`, so the runtime can combine revenue, CRM context, and marketing campaign data without copying them into one warehouse first.


In [ ]:
from pathlib import Path
import subprocess
import sys

import pandas as pd
from IPython.display import display

EXAMPLE_DIR = Path.cwd().resolve()
if not (EXAMPLE_DIR / "langbridge_config.yml").exists():
    candidate = EXAMPLE_DIR / "examples" / "sdk" / "federated_query"
    if candidate.exists():
        EXAMPLE_DIR = candidate.resolve()

REPO_ROOT = EXAMPLE_DIR.parents[2]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from langbridge import LangbridgeClient


## Start the Postgres sources

This example uses Docker to start the sales and CRM databases from the seed scripts in this folder.


In [ ]:
subprocess.run(["docker", "compose", "up", "-d", "--wait"], cwd=EXAMPLE_DIR, check=True)
subprocess.run(["docker", "compose", "ps"], cwd=EXAMPLE_DIR, check=True)


In [ ]:
client = LangbridgeClient.local(config_path=str(EXAMPLE_DIR / "langbridge_config.yml"))
datasets = client.datasets.list()
dataset_ids = {item.name: item.id for item in datasets.items}

datasets_df = pd.DataFrame([item.model_dump(mode="json") for item in datasets.items])
datasets_df = datasets_df.sort_values("name").reset_index(drop=True)
datasets_df


In [ ]:
def preview_dataset(name: str, limit: int = 5) -> pd.DataFrame:
    result = client.datasets.query(dataset_id=dataset_ids[name], limit=limit)
    return pd.DataFrame(result.rows)

for dataset_name in ["sales_orders", "crm_contacts", "marketing_campaigns"]:
    print(dataset_name)
    display(preview_dataset(dataset_name))


## Federated semantic query

This semantic query asks for revenue from the sales database, grouped by CRM account segment and CSV campaign name.


In [ ]:
semantic_result = client.semantic.query(
    "customer_revenue_federation",
    measures=["sales_orders.net_revenue", "sales_orders.order_count"],
    dimensions=["marketing_campaigns.campaign_name", "crm_contacts.account_segment"],
    filters=[{"member": "marketing_campaigns.campaign_name", "operator": "set"}],
    order={"sales_orders.net_revenue": "desc"},
    limit=12,
)

pd.DataFrame(semantic_result.rows)


## Direct federated SQL

Here the runtime federates across the eligible workspace datasets and derives the logical SQL aliases from dataset metadata.


In [ ]:
sql_result = client.sql.query(
    query="""
    SELECT
      m.campaign_name,
      c.account_segment,
      COUNT(DISTINCT s.order_id) AS order_count,
      ROUND(SUM(s.order_total), 2) AS net_revenue
    FROM sales_orders AS s
    JOIN crm_contacts AS c
      ON s.crm_contact_external_id = c.contact_external_id
    JOIN marketing_campaigns AS m
      ON c.contact_external_id = m.contact_external_id
    GROUP BY m.campaign_name, c.account_segment
    ORDER BY net_revenue DESC, order_count DESC
    LIMIT 5
    """,
)

pd.DataFrame(sql_result.rows)


When you are done, run `docker compose down -v` from this folder to remove the demo databases.


In [ ]:
result = client.agents.ask("Which marketing campaign generated the most revenue from enterprise customers? and which month was it?")
print(result)